In [1]:
from pathlib import Path
from torch.utils.data import DataLoader,Dataset
import torch
import h5py
import numpy as np
import re
import torch.nn as nn
import torch.nn.functional as F
import tqdm

In [2]:
N_MASK_CHANNELS = 3

class BraTSSliceDataset(Dataset):
    def __init__(self,files:list[Path]) -> None:
        self.files = list(files)

    def __len__(self) -> int:
        return len(self.files)

    def __getitem__(self,index:int) -> tuple[torch.Tensor,torch.Tensor,str]:
        path = self.files[index]

        with h5py.File(path,"r") as h5_file:
            image = np.asarray(h5_file["image"][...],dtype=np.float32)
            mask = np.asarray(h5_file["mask"][...],dtype=np.float32)

        scale = np.max(np.abs(image), axis=(0,1), keepdims=True)

        scale[scale < 1e-6] = 1.0

        image = np.clip(image / scale, -1.0,1.0)

        image_tensor = torch.from_numpy(image).permute(2,0,1)
        mask_tensor = torch.from_numpy((mask > 0).astype(np.float32)).permute(2,0,1)

        return image_tensor, mask_tensor


In [3]:
def split_by_volume(files:list[Path],seed:int = 42) -> tuple[list[Path], list[Path], list[Path]]:


    volume_pattern = re.compile(r"volume_(\d+)_slice_")
    volume_files: dict[str,list[Path]] = {}

    for path in files:
        match = volume_pattern.search(path.name)
        if match is None:
            raise ValueError(f"Unexpected slice filename: {path.name}")

        volume_files.setdefault(match.group(1),[]).append(path)

        volume_ids = np.asarray(sorted(volume_files.keys()))

        rng = np.random.default_rng(seed)

        rng.shuffle(volume_ids)

        n_volumes = len(volume_ids)
        n_trains = int(n_volumes * 0.70)
        n_val = int(n_volumes * 0.15)

        train_ids = set(volume_ids[:n_trains])
        val_ids = set(volume_ids[n_trains:n_trains+n_val])
        test_ids = set(volume_ids[n_trains + n_val:])

        train = [p for v in train_ids for p in volume_files[v]]
        val = [p for v in val_ids for p in volume_files[v]]
        test = [p for v in test_ids for p in volume_files[v]]


        for group in (train,val,test):
            group.sort()

        if train_ids & val_ids or train_ids & test_ids or val_ids & test_ids:
            raise RuntimeError("Volume split contains data leakage across sets.")

        return train,val,test

        

In [4]:
class DoubleConv(nn.Module):

    def __init__(self,in_channels:int,out_channels:int) -> None:
        super().__init__()

        self.layers = nn.Sequential(
            nn.Conv2d(in_channels,out_channels,kernel_size=3, padding=1,bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels,out_channels,kernel_size=3,padding=1,bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

        def forward(self,x:torch.Tensor) -> torch.Tensor:
            return self.layers(x)



class UNet2D(nn.Module):
    def __init__(self, in_channels:int =4, out_channels:int=3, width:int=8) -> None:
        super().__init__()

        self.pool = nn.MaxPool2d(2)

        self.enc1 = DoubleConv(in_channels,width)
        self.enc2 = DoubleConv(width,width*2)
        self.enc3 = DoubleConv(width*2,width*4)
        self.enc4 = DoubleConv(width*4, width*8)

        self.bottleneck = DoubleConv(width*8,width*16)

        self.up4 = nn.ConvTranspose2d(width*16,width*8,kernel_size=2,stride=2)
        self.dec4 = DoubleConv(width*16,width*8)

        self.up3 = nn.ConvTranspose2d(width*8,width*4,kernel_size=2,stride=2)
        self.dec3 = DoubleConv(width*8,width*4)

        self.up2 = nn.ConvTranspose2d(width*4,width*4,kernel_size=2,stride=2)
        self.dec2 = DoubleConv(width*4,width*2)

        self.up1 = nn.ConvTranspose2d(width*2,width,kernel_size=2,stride=2)
        self.dec1 = DoubleConv(width*2,width)

        self.output = nn.Conv2d(width,out_channels,kernel_size=1)


    def forward(self,x:torch.Tensor) -> tuple[torch.Tensor,torch.Tensor]:

        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        bottleneck = self.bottleneck(self.pool(e4))

        d4 = self.dec4(torch.cat([self.up4(bottleneck),e4],dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4),e3],dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3),e2],dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2),e1],dim=1))

        return self.output(d1),bottleneck
    

In [ ]:
def mask_positive_weights(files: list[Path],device:torch.device) -> torch.Tensor:
    positives = np.zeros(N_MASK_CHANNELS, dtype=np.float64)
    total_pixels = 0

    for path in tqdm(files,desc="Counting training mask pixels", unit="slice"):
        with h5py.File(path,"r") as h5_file:
            mask = np.asarray(h5_file["mask"][...]) > 0

        positives += mask.sum(axis=(0,1))
        total_pixels += mask.shape[0] * mask.shape[1]

    negatives = total_pixels - positives

    weights = negatives / np.maximum(positives,1.0)

    return torch.as_tensor(np.clip(weights,1.0,20.0),dtype=torch.float32,device=device)

def segmentation_loss(logits: torch.Tensor,masks:torch.Tensor, pos_weight:torch.Tensor) -> torch.Tensor:

    bce = F.binary_cross_entropy_with_logits(
        logits,
        masks,
        pos_weight=pos_weight.view(1,-1,1,1)
        reduction="none"
    ).mean(dim=(2,3))

    probabilities = torch.sigmoid(logits)

    intersection = (probabilities * masks).sum(dim=(2,3))
    denominator = probabilities.sum(dim=(2,3)) + masks.sum(dim=(2,3))

    dice_loss = 1.0 - (2.0 * intersection + 1.0) / (denominator + 1.0)

    per_sample_class_loss = bce + dice_loss
    return per_sample_class_loss.mean()


